# 00 — Setup and real-world data download

This notebook prepares the Colab runtime and downloads real OHLCV data for the demo.

**Default universe**
- `RELIANCE.NS`
- `TCS.NS`
- `INFY.NS`
- benchmark: `^NSEI`

You can edit the ticker list before running the download cell.

In [ ]:
# In Colab, clone the repo. If you already cloned it, this cell is safe to skip.
REPO_URL = "https://github.com/Randhir123/nifty50-multimodal-transformer.git"
REPO_DIR = "/content/nifty50-multimodal-transformer"

import os, subprocess, pathlib
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies.
# Restart runtime only if Colab asks you to after installing packages.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Installed project dependencies.")

In [ ]:
from datetime import date, timedelta
from pathlib import Path
import pandas as pd

from src.data.download_yfinance import (
    download_multiple_tickers,
    download_benchmark_data,
    save_ticker_csv,
)

TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS"]
BENCHMARK = "^NSEI"
END = date.today().isoformat()
START = (date.today() - timedelta(days=31 * 9)).isoformat()

OUTPUT_DIR = Path("data/processed/real_world_demo")
RAW_DIR = OUTPUT_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Tickers:", TICKERS)
print("Benchmark:", BENCHMARK)
print("Range:", START, "to", END)

In [ ]:
# Download stock and benchmark OHLCV snapshots from yfinance and cache them as CSV.
stock_data = download_multiple_tickers(TICKERS, start=START, end=END)
for ticker, df in stock_data.items():
    path = save_ticker_csv(ticker, df, output_dir=RAW_DIR)
    print(ticker, "->", path, df.shape)

benchmark_df = download_benchmark_data(BENCHMARK, start=START, end=END)
benchmark_path = save_ticker_csv(BENCHMARK, benchmark_df, output_dir=RAW_DIR)
print(BENCHMARK, "->", benchmark_path, benchmark_df.shape)

In [ ]:
# Quick preview of one stock and the benchmark.
display(stock_data[TICKERS[0]].head())
display(benchmark_df.head())

## Checkpoint output

After this notebook, you should have CSV snapshots under:

```text
data/processed/real_world_demo/raw/
```

Next notebook: `01_features_labels_and_alignment.ipynb`.